In [1]:
import os

In [2]:
# rapid fire imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)

In [4]:
# document loaders
from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredRSTLoader
from langchain_core.documents import Document

# text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# retriever
from langchain_core.retrievers import BaseRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import TFIDFRetriever, AmazonKnowledgeBasesRetriever

# cross encoder for reranking
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker


[https://oss-docs.rapidfire.ai/en/latest/ragspecs.html](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

In [5]:
class UnstructuredRSTLoaderWithMetadata(UnstructuredRSTLoader):
    def __init__(self, file_path: str, metadata_func=None, **kwargs):
        super().__init__(file_path=file_path, **kwargs)
        self.metadata_func = metadata_func
    
    def load(self) -> list[Document]:
        # load the documents as UnstructuredRSTLoader
        docs = super.load()
        # if a metadata function is specified
        if self.metadata_func is not None:
            for d in docs:
                d.metadata = self.metadata_func(d, d.metadata)
        
        return docs
    
    def lazy_load(self):
        # for each document loaded using UnstructuredRSTLoader
        for doc in super().lazy_load():
            # if a metadata function is specified
            if self.metadata_func is not None:
                # apply the metadata function to the document
                doc.metadata = self.metadata_func(doc, dict(doc.metadata))
            yield doc

In [6]:
def metadata_func(document, metadata):
    # extract the page title from the content and store it in metadata
    title = document.page_content.split('\n')[0].split('===')[0]
    metadata['title'] = title
    return metadata

In [7]:
loader = DirectoryLoader(
    path='../data/sourcedocs/',
    glob='**/*.rst',
    loader_cls=UnstructuredRSTLoaderWithMetadata,
    loader_kwargs={
        'metadata_func': metadata_func,
        'mode': 'single',
    },
    show_progress=True,
    use_multithreading=True
)

In [8]:
docs = loader.load()

 35%|███▌      | 11/31 [00:02<00:03,  6.45it/s][WARNING] Could not convert TeX math \text{FPC} = \sqrt{(N-n)/(N-1)}, rendering as TeX
[WARNING] Could not convert TeX math \text{SE}_{\hat{p}} = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \text{center} = \frac{\hat{p} + z^2/(2n_{\text{eff}})}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \text{margin} = \frac{z\sqrt{\hat{p}(1-\hat{p})/n_{\text{eff}} + z^2/(4n_{\text{eff}}^2)}}{1 + z^2/n_{\text{eff}}}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon = (b-a)\sqrt{\frac{\ln(2/\alpha)}{2n}} \times \text{FPC}, rendering as TeX
[WARNING] Could not convert TeX math \varepsilon_{\bar{X}} = R\sqrt{\ln(2/\alpha)/(2n)}, rendering as TeX

100%|██████████| 31/31 [00:04<00:00,  6.75it/s]


Models:
- `gpt2` / `r50k_base`: older ~50k-vocab GPT-2 style tokenizers. `r50k_base` is the standard GPT-2 BPE; `gpt2` is the GPT-2 encoding entry in `tiktoken`. Both use the same regex pattern and the same end-of-text token, with `gpt2` built from the original GPT-2 encoder/vocab files
- `p50k_base`: also ~50k vocab, but different merge ranks from `r50k_base`; intended for Codex-style/code-heavy tokenization
- `p50k_edit`: same base merges as `p50k_base`, but adds extra FIM/edit special tokens like `<|fim_prefix|>`, `<|fim_middle|>`, and `<|fim_suffix|>`
- `cl100k_base`: newer ~100k-vocab tokenizer with a different regex and a larger special-token set; commonly associated with GPT-4 / GPT-3.5-era model families
- `o200k_base`: newer ~200k-vocab tokenizer with a different regex again; used by newer model families such as `gpt-4o`
- `o200k_harmony`: derived from `o200k_base` but with many additional special tokens layered on top

In [9]:
# text splitter 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
     encoding_name='o200k_base',
     chunk_size=512, 
     chunk_overlap=32
)

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`embedding_cfg (dict[str, Any], optional)`

The embedding class and its kwargs to convert a chunk/query into a vector, provided as a single dictionary. Must include a key `"class"` with the class itself as value, not an instance. Options for the class include `HuggingFaceEmbeddings` and `OpenAIEmbeddings`. 

The kwargs that follow must contain all parameters needed to initialize the embedding class; required parameters vary by embedding class. For example, `HuggingFaceEmbeddings` needs `model_name`, `model_kwargs` and `device`, while `OpenAIEmbeddings` needs `"model"` and `"api_key"`.

In [10]:
batch_size = 32

embedding_cfg = List([
    # yet to use open ai embeddings
    {
        'class': HuggingFaceEmbeddings,
        'model_name': 'sentence-transformers/all-MiniLM-L6-v2',
        'model_kwargs': {'device': 'cpu'},
        'encode_kwargs': {'normalize_embeddings': True, 'batch_size': batch_size}
    }
])

From [RapidFire AI oss](https://oss-docs.rapidfire.ai/en/latest/ragspecs.html)

`vector_store_cfg (dict[str, Any], optional)`

The vector store type and args to store and possibly index embedding vectors for retrieval, provided as a single dictionary.

- `"type"`: The type of vector store to use. Must be one of `"faiss"`, `"pgvector"`, or `"pinecone"`. Required.
- `"batch_size"`: Number of vectors per insert batch. Applies to all 3 types of stores. Optional; default is 128.

In [11]:
vector_store_cfg={
    'type': 'faiss', 
    'batch_size': batch_size
}

In [12]:
search_cfg=List([
    {'type': 'similarity', 'k': 10},
    {'type': 'mmr', 'k': 10}
])

In [16]:
reranker_cfg={
    'class': CrossEncoderReranker,
    'model_name': 'cross-encoder/ms-marco-MiniLM-L6-v2',
    'model_kwargs': {'device': 'cpu'},
    'top_n': 5,
}

In [17]:
def custom_template(doc):
    return f'{doc.metadata["title"]}: {doc.page_content}'

In [18]:
rag = RFLangChainRagSpec(
    # to load the corpus
    document_loader=loader,
    # to split documents into chunks
    text_splitter=text_splitter,
    # to embed the chunks
    embedding_cfg=embedding_cfg,
    # vectore storage
    vector_store_cfg=vector_store_cfg,
    # search algorithm
    search_cfg=search_cfg,
    # rerank top results with a cross encoder
    reranker_cfg=reranker_cfg,
    # whether to use gpu-accelerated faiss
    enable_gpu_search=False,
    # to format each retrieved chunk for context injection into prompts
    document_template=custom_template,
)